# 00 — Project overview

**Audience:** complete beginner. We will *not* assume you know what a neural network is.

This series of notebooks teaches the **car-crash-fix-amount-predictor** project from first principles. By the end you will be able to:

1. Explain what every model in the pipeline does and why it exists.
2. Read and modify the training code.
3. Run a small training job on Google Colab.
4. Understand every number in the evaluation report.

## The problem we are solving

> Given a photo of a damaged car, predict (a) what kind of damage it is, (b) where on the car it is, and (c) roughly how much it will cost to repair.

A real insurance assessor does this with a clipboard and 15 years of experience. We will do it with three machine-learning models stacked together.

## The pipeline at a glance

```
    [User uploads photo]
            |
            v
    +----------------+
    |  Preprocess    |   Downscale + quality score (no ML)
    +----------------+
            |
            +----------------------+----------------------+
            v                      v                      v
    +-------------+        +-------------+        +-------------+
    |  ResNet50   |        |   YOLOv8    |        |  ResNet50   |
    |  classifier |        |  detector   |        |  identifier |
    |  (what?)    |        |  (where?)   |        | (car make?) |
    +-------------+        +-------------+        +-------------+
            \                    /                    /
             \                  /                    /
              v                v                    v
            +------------------------------------+
            |  XGBoost cost regressor (how much?)|
            +------------------------------------+
                            |
                            v
                  +-----------------+
                  | Catalog lookup  |   Fallback if XGBoost can't trust input
                  +-----------------+
                            |
                            v
                    Final $ estimate
```

## What each notebook covers

| # | Notebook | What you'll learn |
|---|---|---|
| 01 | Data & preprocessing | Where the data comes from, what cleaning is needed, why we downscale, how to measure blur |
| 02 | ResNet50 classifier | What a CNN is, why ResNet is *residual*, two-stage transfer learning |
| 03 | YOLOv8 detector | How a model predicts boxes, IoU, NMS, mAP — all with diagrams |
| 04 | XGBoost cost regressor | What gradient boosting is, our calibration trick |
| 05 | Metrics deep dive | Precision, recall, F1, mAP, RMSE, MAE, MAPE, R² with worked examples |
| 06 | End-to-end inference | Load the production weights and run a real prediction |

Run them in order — each builds on the last.


In [ ]:
# === Colab bootstrap ===
# Safe to re-run. On a local clone with `pip install -e .` already done this
# is a no-op; on Colab it clones the repo + installs deps the first time.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/theDocWho/car-crash-fix-amount-predictor.git"
REPO_DIR = Path("car-crash-fix-amount-predictor")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
if IN_COLAB:
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "requirements.txt"], check=True)

# Make `ccdp` importable whether or not the package was installed editable.
src_path = Path("src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("ccdp path:", src_path)
print("running in Colab:", IN_COLAB)


## Verify the install

The cell below imports the project and prints which models are reachable. If any line errors, the bootstrap above did not finish — re-run it.


In [ ]:
# Smoke-test imports — if these all succeed, the project is wired up correctly.
import ccdp
from ccdp.preprocess import preprocess
from ccdp.data.schema import DAMAGE_TYPES
from ccdp.viz import annotate_prediction

print("DAMAGE_TYPES the models can output:", DAMAGE_TYPES)
print()
print("ccdp imported from:", Path(ccdp.__file__).parent)


## Visualising the data flow as a diagram

Below is the same pipeline drawn with `matplotlib`. We will redraw versions of this diagram in every notebook so you build a mental picture of how the pieces fit.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(10, 5))
ax.set_xlim(0, 10); ax.set_ylim(0, 5); ax.axis("off")

boxes = [
    (0.5, 2,  "Image",        "#bbdefb"),
    (2.2, 2,  "Preprocess",   "#c8e6c9"),
    (4.0, 3.5,"ResNet50\nclassifier", "#fff59d"),
    (4.0, 2,  "YOLOv8\ndetector",    "#ffcc80"),
    (4.0, 0.5,"ResNet50\nidentifier","#ce93d8"),
    (6.5, 2,  "XGBoost\ncost",       "#ef9a9a"),
    (8.5, 2,  "$ estimate",          "#80cbc4"),
]
for x, y, t, c in boxes:
    ax.add_patch(patches.FancyBboxPatch((x, y-0.4), 1.4, 0.8,
                 boxstyle="round,pad=0.05", facecolor=c, edgecolor="black"))
    ax.text(x+0.7, y, t, ha="center", va="center", fontsize=10)
# arrows
for (x1, y1), (x2, y2) in [
    ((1.9, 2),(2.2,2)), ((3.6,2),(4.0,2)),
    ((3.6,2),(4.0,3.5)),((3.6,2),(4.0,0.5)),
    ((5.4,3.5),(6.5,2.2)),((5.4,2),(6.5,2)),((5.4,0.5),(6.5,1.8)),
    ((7.9,2),(8.5,2)),
]:
    ax.annotate("", xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle="->", lw=1.2))
plt.title("ccdp pipeline — what each notebook covers")
plt.show()


**Next:** open `01_data_and_preprocessing.ipynb` to learn where the training data comes from and why preprocessing matters before you ever touch a neural network.
